In [ ]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u

Using the regridded FITS files copied and renamed from 'test7', with CHIME in Jy/beam and DRAGONS in K.

### Make spatially filtered versions of the maps and save FITS cubes

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'data/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes+'.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_'+stokes+'.fits'

    print(chime_file)
    print(dragons_file)
    dragons_chime_conv_combine.band_pass(chime_file, dragons_file, dir_out+sub_dir,stokes)

### Calculate the Jy/beam to K conversion factors for CHIME, NOT spatially filtered

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_OG/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes+'.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_'+stokes+'.fits'

    dragons_chime_conv_combine.compare_chime_dragons(chime_file, dragons_file, dir_out+sub_dir,stokes)

# Organize the TT plots:

freqs_new = np.linspace(512.5,729.5,218)
for freq in freqs_new[0:218]:
    !{'mkdir '+dir_out+sub_dir+str(int(np.floor(freq)))}
    !{'mv '+dir_out+sub_dir+'*'+str(freq)+'*.png '+dir_out+sub_dir+str(int(np.floor(freq)))+'/'}

### Calculate the Jy/beam to K conversion factors for CHIME using spatially filtered data

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_filtered_v2/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_filtered_'+stokes+'_v2.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_filtered_'+stokes+'_v2.fits'

    dragons_chime_conv_combine.compare_chime_dragons(chime_file, dragons_file, dir_out+sub_dir,stokes)

# Organize the TT plots:

freqs_new = np.linspace(512.5,729.5,218)
for freq in freqs_new[0:218]:
    !{'mkdir '+dir_out+sub_dir+str(int(np.floor(freq)))}
    !{'mv '+dir_out+sub_dir+'*'+str(freq)+'*.png '+dir_out+sub_dir+str(int(np.floor(freq)))+'/'}

### Plots of conversion factor versus declination and frequency

In [ ]:
def conversion_vs_freq(idx,paramsQ,paramsU):

    fs = 14
    
    print(paramsQ['decs'][idx])

    fig,ax = plt.subplots(2,1,figsize=(16,8))

    ax[0].scatter(paramsQ['freqs'],paramsQ['slopes'][:,idx],label='Stokes Q',s=20)
    ax[0].scatter(paramsU['freqs'],paramsU['slopes'][:,idx],label='Stokes U',s=20)
    ax[0].set_ylim(0,3)
    ax[0].set_ylabel('Slope [K/Jy/beam]',fontsize=fs)

    ax[1].scatter(paramsQ['freqs'],paramsQ['ints'][:,idx],label='Stokes Q',s=20)
    ax[1].scatter(paramsU['freqs'],paramsU['ints'][:,idx],label='Stokes U',s=20)
    ax[1].set_ylim(-0.5,0.5)
    ax[1].set_ylabel('Intercept [K]',fontsize=fs)

    for i in range(0,2):
        ax[i].set_xlim(510,730)
        ax[i].grid()
        ax[i].set_xlabel('Frequency',fontsize=fs)
        ax[i].legend(fontsize=fs)
        ax[i].tick_params(axis='both', labelsize=fs)

    dec_string = str(paramsQ['decs'][idx])
    plt.suptitle('DRAGONS/CHIME conversion factor, dec='+dec_string,fontsize=fs+4)
    
    return


def conversion_vs_dec(idx,paramsQ,paramsU):

    fs = 14
    
    print(paramsQ['freqs'][idx])

    fig,ax = plt.subplots(2,1,figsize=(16,8))

    ax[0].scatter(paramsQ['decs'],paramsQ['slopes'][idx,:],label='Stokes Q',s=10)
    ax[0].scatter(paramsU['decs'],paramsU['slopes'][idx,:],label='Stokes U',s=10)
    ax[0].set_ylim(0,3)
    ax[0].set_ylabel('Slope [K/Jy/beam]',fontsize=fs)

    ax[1].scatter(paramsQ['decs'],paramsQ['ints'][idx,:],label='Stokes Q',s=10)
    ax[1].scatter(paramsU['decs'],paramsU['ints'][idx,:],label='Stokes U',s=10)
    ax[1].set_ylim(-0.5,0.5)
    ax[1].set_ylabel('Intercept [K]',fontsize=fs)

    for i in range(0,2):
        ax[i].set_xlim(10,90)
        ax[i].grid()
        ax[i].set_xlabel('Declination',fontsize=fs)
        ax[i].legend(fontsize=fs)
        ax[i].tick_params(axis='both', labelsize=fs)

    freq_string = str(paramsQ['freqs'][idx])
    plt.suptitle('DRAGONS/CHIME conversion factor, '+freq_string+' MHz',fontsize=fs+4)
        
    return 

In [ ]:
paramsQ = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_v2/chime_vs_dragons_stokesQ.npz')
paramsU = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_v2/chime_vs_dragons_stokesU.npz')
print(paramsQ.keys())
print(paramsU.keys())

print(paramsQ['decs'].shape)
print(paramsQ['slopes'].shape)
print(paramsQ['freqs'].shape)
print(paramsQ['ints'].shape)

In [ ]:
for idx in range(0,218):

    try:
        conversion_vs_dec(idx,paramsQ,paramsU)
        freq_string = str(paramsQ['freqs'][idx])
        plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_v2/conversion_plots/conversion_vs_dec_'+freq_string+'.png',
                    facecolor='white')
    except:
        pass

In [ ]:
idx_min = 801
idx_max = 1412
print(paramsQ['decs'][idx_min])
print(paramsQ['decs'][idx_max])

for idx in range(idx_min,idx_max):

    try:
        conversion_vs_freq(idx,paramsQ,paramsU)
        dec_string = str(paramsQ['decs'][idx])
        plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_v2/conversion_plots/conversion_vs_freq_'+dec_string+'.png')
    except:
        pass


### Applying conversion factor to CHIME

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_filtered/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes+'.fits'
    conversion_file = '/srv/data/gmims/gmims-lbn/experiments/results_filtered/chime_vs_dragons_stokes'+stokes+'.npz'

    dragons_chime_conv_combine.chimeK(chime_file, conversion_file, dir_out+sub_dir, stokes)

In [ ]:
def read_in_maps(file1,file2):

    hdu  = fits.open(file1)
    map1 = hdu[0].data

    hdu  = fits.open(file2)
    map2 = hdu[0].data   
    
    wcs = WCS(hdu[0].header)

    freqs = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6

    return map1, map2, wcs, freqs

In [ ]:
def make_diff_map(wcs, map1, map2, stokes = 'pi',
                  vmax=20, diff_frac=0.5, RA_range = [24,0],dec_range = [-30,90]):

    crange = SkyCoord(RA_range, dec_range, frame=ICRS, unit=(u.hourangle, u.deg))
    
    fs = 14
    width = 20
    
    height = 2.5*width*(dec_range[1]-dec_range[0])/((RA_range[0]-RA_range[1])*360/24)
    print(height)
    
    fig = plt.figure(figsize=(width,height))
    ax1 = fig.add_subplot(311, projection=wcs.celestial)
    ax2 = fig.add_subplot(312, projection=wcs.celestial)
    ax3 = fig.add_subplot(313, projection=wcs.celestial)

    if stokes == 'i':
        vmin = 0
        cmap = 'cubehelix'
    if stokes == 'pi':
        vmin = 0
        cmap = 'viridis'
    if ((stokes == 'q') or (stokes == 'u')):
        vmin = -vmax
        cmap = 'RdBu_r'

    im1 = ax1.imshow(map1,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im2 = ax2.imshow(map2,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im3 = ax3.imshow(map2-map1,origin='lower',cmap='RdBu_r',vmin=-vmax*diff_frac,vmax=vmax*diff_frac)
 
    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]
    
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=1, pad=0.01)
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('dec',fontsize=fs)
    ax3.set_xlabel('RA',fontsize=fs)

    return

In [ ]:
stokes = 'q'

file1 = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes.upper()+'.fits'
#file1 = '/srv/data/gmims/gmims-lbn/experiments/results_OG/chime_K_'+stokes.upper()+'.fits'
file2 = '/srv/data/gmims/gmims-lbn/experiments/results_filtered/chime_K_'+stokes.upper()+'.fits'

map1, map2, wcs, freqs = read_in_maps(file1,file2)

print(map1.shape)
print(map2.shape)

In [ ]:
fpick = 690.5

fidx = abs(freqs-fpick).argmin()
print(fidx,freqs[fidx])

make_diff_map(wcs.dropaxis(2), map1[fidx], map2[fidx], stokes = stokes,
              vmax=1, diff_frac=0.5, RA_range = [8,0],dec_range = [30,75])